# Parcel statistics from HUB Sentinel ARD

Read selected physical-band windows from `ARD_S2` and/or `ARD_S1`, then use the
shared local cleaning, index and parcel-statistics engine. This notebook does
not produce ARD or submit openEO jobs. HUB S1 gamma0 and openEO sigma0 have
different calibration; retain source provenance when training or predicting.

Use the repository's editable installation with the `acquisition` and `notebooks`
extras available in this kernel. See the [HUB guide](../../../docs/data_preparation/hub/README.md).
The notebook is supplied without executed cells or saved credentials.


## Load local settings before catalog imports

Open this notebook from the repository or one of its subdirectories. Existing
process environment settings take precedence over `.env`. Only the presence of
the four required settings is checked; their values are not displayed.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src" / "data_preparation").is_dir()
)
load_dotenv(REPO_ROOT / ".env", override=False)
required_settings = ("CATALOG_URL", "KEYCLOAK_URL", "CATALOG_CLIENT_ID", "CATALOG_CLIENT_SECRET")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise ValueError(f"Error: Missing HUB settings: {', '.join(missing_settings)}")

import geopandas as gpd
import pandas as pd

from data_preparation.parcel_stats import HubZonalStats
from shared.io import read_data


## Choose parcels, dates, and a sensor configuration

Edit `PARCELS_PATH` to your polygon dataset. The initial path follows the Neuro
project's existing `WORK_PATH/neuro_2024-2025/sample_parcels.gpkg` convention.
Start with five nearby parcels and a short interval with known HUB coverage.
Switch `SENSOR_MODE` between `s2`, `s1`, and `combined`; each gets its own output
folder. Inputs remain outside that folder. S2-only is the initial smoke test.


In [ ]:
WORK_PATH = Path(os.getenv("WORK_PATH") or (REPO_ROOT / "data" / "raw"))
PARCELS_PATH = Path("C:/work_sat/central_greece/cleaned_grouped_parcels_20261124_094211.gpkg")
START_DATE = "2025-07-01"
END_DATE = "2025-07-31"
SENSOR_MODE = "s2"
PARCEL_LIMIT = 5

sensor_options = {
    "s2": {"sentinel2_bands": ["B04", "B08"], "sentinel2_indices": ["NDVI"],
           "sentinel1_bands": [], "sentinel1_indices": []},
    "s1": {"sentinel2_bands": [], "sentinel2_indices": [],
           "sentinel1_bands": ["VV", "VH"], "sentinel1_indices": ["R", "RVI"]},
    "combined": {"sentinel2_bands": ["B04", "B08"], "sentinel2_indices": ["NDVI"],
                 "sentinel1_bands": ["VV", "VH"], "sentinel1_indices": ["R", "RVI"]},
}
if SENSOR_MODE not in sensor_options:
    raise ValueError("Error: SENSOR_MODE must be s2, s1, or combined.")
OUTPUT_DIR = REPO_ROOT / "outputs" / "neuro" / "hub_ard" / f"{START_DATE}_{END_DATE}_{SENSOR_MODE}"


In [ ]:
if not PARCELS_PATH.is_file():
    raise FileNotFoundError(f"Error: Set PARCELS_PATH to your parcel dataset: {PARCELS_PATH}")
parcels = read_data(str(PARCELS_PATH))
if not isinstance(parcels, gpd.GeoDataFrame) or parcels.crs is None:
    raise ValueError("Error: Parcel input must contain polygon geometry and a CRS.")
parcels = parcels.head(PARCEL_LIMIT).copy()
if parcels.empty:
    raise ValueError("Error: Parcel input is empty.")
print(f"Selected {len(parcels)} parcels; output directory: {OUTPUT_DIR}")
display(parcels.head())


## Configure and run

The initial run keeps acquisition timestamps (`temporal_reducer="none"`) and
reports monthly statistics. Cleaning and filling are disabled for a transparent
first test; enable them after inspecting the results. `batch_workers=1` keeps the
first test small. Compatible completed outputs can be resumed; source discovery
still refreshes the manifest before completed-output reuse.

The next cell acquires data and writes results when you execute it. No cells in
this notebook were run during authoring. Adjust dates if the selected area has
no eligible ARD items. HTTP gateway redirects and storage range reads are handled
by the HUB adapter.


In [ ]:
pipeline = HubZonalStats(
    parcels=parcels,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
    working_epsg=2100,
    **sensor_options[SENSOR_MODE],
    sentinel1_orbit_direction="BOTH",
    temporal_period="1M",
    temporal_reducer="none",
    spatial_statistics=["mean", "median", "sd", "min", "max", "range"],
    remove_outliers=False,
    fill_nulls=False,
    batch_workers=1,
    remove_nc_after_completion=False,
    resume_completed_partitions=True,
)
results = pipeline.run()


## Inspect persisted outputs

The main GeoParquet has one row per parcel, with dated features. The reduced
GeoParquet summarizes temporal medians. The cleaning CSV records local
processing diagnostics. Inspect the HUB manifest and provenance alongside the
values before comparing sources or using these features in a model.


In [ ]:
if len(results) != len(parcels) or results[pipeline.PARCEL_ID_FIELD].duplicated().any():
    raise ValueError("Error: Expected exactly one output row per selected parcel.")
feature_columns = [column for column in results.columns if "__" in column]
print(f"Result: {len(results)} parcels, {len(feature_columns)} dated feature columns")
display(results.head())
display(results[feature_columns].isna().mean().rename("missing_fraction").to_frame())
display(pipeline.cleaning_report.head())

saved = gpd.read_parquet(OUTPUT_DIR / pipeline.PARCEL_OUTPUT_FILE_NAME)
reduced = gpd.read_parquet(OUTPUT_DIR / pipeline.REDUCED_PARCEL_OUTPUT_FILE_NAME)
pd.testing.assert_frame_equal(saved, results)
print(f"Verified saved output; reduced table shape: {reduced.shape}")
print("Review manifest and completion/provenance files in the output directory.")


## After the test

Review missingness, the item inventory, and local cleaning diagnostics before
expanding the date range or parcel count. To test another sensor configuration,
change `SENSOR_MODE` and rerun the configuration and subsequent cells; its output
folder is separate. Reuse a configuration unchanged to test resume behavior.
No accuracy or coverage conclusion is made until this notebook is executed with
your parcels and the resulting outputs are inspected.
